# GAN-Based Synthetic Device Fault-Code Image Generation

**Domain:** Appliance warranty remote diagnostics (WarrantyGraph)
**Goal:** Synthetically generate *claim-photo-like* images of machine-displayed fault/error codes so we can train and evaluate OCR → graph lookup when historical claim photos are sparse or missing.
**Stack:** PyTorch DCGAN + Conditional GAN (cGAN), PIL seed renderer, Neo4j Cypher templates aligned with this repo.

---

## Why this exists (client problem)

| Reality | Impact |
|---------|--------|
| Customers photograph the **error code on the appliance display** when filing a claim | Our app must **read** that code (OCR / vision) |
| Extracted code maps to `ErrorCode` nodes and boosts failure-mode ranking | GraphRAG Cypher: `(:ErrorCode)-[:INDICATES]->(:FailureMode)` |
| Historical resolution cases often **lack** those photos | Sparse / zero real images → cannot measure OCR accuracy or end-to-end efficacy |
| Need synthetic images that **mimic** LCD/LED fault displays under phone-camera conditions | Train & stress-test extraction **before** real claim volume exists |

```
Customer phone photo --> OCR / CV --> fault code string
                                              |
                                              v
                    MATCH (p:Product)-[:HAS_ERROR_CODE]->(ec:ErrorCode {code:$code})
                          (ec)-[:INDICATES]->(fm:FailureMode)
                                              |
                                              v
                         ranked FM + CONFIRMS steps + resolution path
```

This notebook is **self-contained**: it does **not** require Neo4j or GPU (Apple MPS / CUDA used if present). It produces images under `notebooks/fault_code_gan_artifacts/`.

---

## Table of contents

| Part | Content |
|------|---------|
| **0** | Setup & domain fault-code catalog |
| **1** | GAN theory (Goodfellow, DCGAN, cGAN) — authoritative sources |
| **2** | Procedural *seed* LCD/LED display images (bootstrap when no real photos) |
| **3** | Dataset, classical augmentations (phone-camera realism) |
| **4** | DCGAN implementation & training |
| **5** | Conditional GAN (class = fault code) |
| **6** | Generate synthetic claim-photo corpus |
| **7** | Lightweight OCR / code extraction eval |
| **8** | Cypher bridge → Neo4j GraphRAG (this codebase) |
| **9** | References & production notes |

---

## Authoritative sources (theory + practice)

| Source | Contribution used here |
|--------|-------------------------|
| Goodfellow et al., *Generative Adversarial Nets*, NeurIPS 2014 | Minimax game \(G\) vs \(D\); value function \(V(D,G)\) |
| Radford, Metz, Chintala, *Unsupervised Representation Learning with Deep Convolutional GANs* (DCGAN), ICLR 2016 | Conv Generator/Discriminator recipes (BatchNorm, LeakyReLU, strided conv, tanh) |
| Mirza & Osindero, *Conditional Generative Adversarial Nets*, 2014 | Class conditioning \(p(x \mid y)\) — generate image **of a chosen fault code** |
| Shorten & Khoshgoftaar, *A survey on Image Data Augmentation for Deep Learning*, J. Big Data 2019 | Geometric/photometric augments as complementary to generative models |
| [PyTorch DCGAN tutorial](https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html) | Practical training loop patterns |
| Ho et al. DDPM 2020; Rombach et al. LDM 2022 | *Beyond* GANs for higher fidelity (upgrade path) |
| This repo | `graph/graph_rag.py` (`match_error_codes`, `rank_failure_modes_with_error_codes`), `graph/oem_product_catalog.py` error codes |

> **Honest framing:** When *no* real photos exist, industry practice is **procedural simulation + generative models** (GANs/diffusion) to create a labelled corpus, then **validate OCR on a hold-out of real photos** as soon as any arrive. Do not train production OCR *only* on synthetic data without a real-image gate (see study module 20 in this repo).

## Part 0 — Environment setup

Core path uses **torch**, **torchvision**, **Pillow**, **matplotlib**, **numpy** (already common in this workspace). No paid APIs required.

In [ ]:
from __future__ import annotations

import os
import random
import re
import json
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print(f"torch {torch.__version__} | device = {DEVICE}")

ROOT = Path(".").resolve()
# Notebook may be launched from repo root or notebooks/
if (ROOT / "notebooks").is_dir() and not (ROOT / "fault_code_gan_artifacts").is_dir():
    ART = ROOT / "notebooks" / "fault_code_gan_artifacts"
elif (ROOT / "fault_code_gan_artifacts").is_dir() or ROOT.name == "notebooks":
    ART = ROOT / "fault_code_gan_artifacts"
else:
    ART = ROOT / "notebooks" / "fault_code_gan_artifacts"
ART.mkdir(parents=True, exist_ok=True)
(SEED_DIR := ART / "seed_lcd").mkdir(exist_ok=True)
(AUG_DIR := ART / "augmented").mkdir(exist_ok=True)
(GEN_DIR := ART / "generated").mkdir(exist_ok=True)
(CKPT_DIR := ART / "checkpoints").mkdir(exist_ok=True)
print(f"Artifacts -> {ART}")

### Domain fault-code catalog

Codes mirror `graph/oem_product_catalog.py` (washer/dryer/fridge-style OEM codes used in the demo graph). Each code later becomes a **cGAN class label** and a Cypher lookup key.

In [ ]:
# Codes drawn from graph/oem_product_catalog.py (representative OEM fault strings)
FAULT_CATALOG: list[dict] = [
    {"code": "UE", "description": "Unbalanced load — rebalance or calibration", "family": "washer"},
    {"code": "4E", "description": "Water supply error — inlet valve or pressure", "family": "washer"},
    {"code": "5E", "description": "Drain error — filter, hose, or pump", "family": "washer"},
    {"code": "OE", "description": "Drain timeout / overfill related", "family": "washer"},
    {"code": "IE", "description": "Water fill insufficient after 10 minutes", "family": "washer"},
    {"code": "HE", "description": "Heating error", "family": "washer"},
    {"code": "AE", "description": "Leak / overfill protection activated", "family": "washer"},
    {"code": "E24", "description": "Drain issue", "family": "dishwasher"},
    {"code": "E01", "description": "Pump failure", "family": "dishwasher"},
    {"code": "F9E1", "description": "Drain pump fault", "family": "dryer"},
    {"code": "F9E0", "description": "Drain system fault variant", "family": "dryer"},
    {"code": "22E", "description": "Evaporator fan error", "family": "fridge"},
    {"code": "33E", "description": "Ice maker fan error", "family": "fridge"},
    {"code": "39E", "description": "Ice maker function error", "family": "fridge"},
    {"code": "LC", "description": "Child lock / control lock", "family": "washer"},
    {"code": "OC", "description": "Over current / overload", "family": "generic"},
    {"code": "DE", "description": "Door error", "family": "washer"},
]

CODE_TO_IDX = {row["code"]: i for i, row in enumerate(FAULT_CATALOG)}
IDX_TO_CODE = {i: row["code"] for i, row in enumerate(FAULT_CATALOG)}
N_CLASSES = len(FAULT_CATALOG)
print(f"{N_CLASSES} fault codes:", ", ".join(IDX_TO_CODE[i] for i in range(N_CLASSES)))

## Part 1 — GAN principles (authoritative theory)

### 1.1 The adversarial game (Goodfellow et al., 2014)

Two networks train jointly:

- **Generator** \(G(z)\): maps noise \(z \sim p_z\) (usually \(\mathcal{N}(0,I)\)) to a fake sample \(\tilde{x}\).
- **Discriminator** \(D(x)\): outputs probability that \(x\) is real.

Value function:

\[
\min_G \max_D V(D,G)
= \mathbb{E}_{x \sim p_{\mathrm{data}}}[\log D(x)]
+ \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]
\]

**Intuition:** \(D\) learns to tell real LCD photos from fakes; \(G\) learns to fool \(D\). At equilibrium, \(G\) samples from the data distribution.

**Training practice (non-saturating heuristic):** train \(G\) to **maximize** \(\log D(G(z))\) instead of minimize \(\log(1-D(G(z)))\) — gradients stay informative early in training (Goodfellow 2014, §3).

### 1.2 DCGAN architectural constraints (Radford et al., 2016)

| Rule | Implementation |
|------|----------------|
| No pooling — use strided conv / fractionally-strided conv | `Conv2d` / `ConvTranspose2d` |
| BatchNorm in \(G\) and \(D\) (except \(G\) output & \(D\) input) | `BatchNorm2d` |
| ReLU in \(G\) (except `Tanh` output); LeakyReLU in \(D\) | `nn.ReLU`, `nn.LeakyReLU(0.2)` |
| Normalize inputs to \([-1,1]\) for `Tanh` | `transforms.Normalize(0.5, 0.5)` |

### 1.3 Conditional GAN (Mirza & Osindero, 2014)

We need images of a **specific** fault code (`5E` vs `UE`). Condition both networks on label \(y\):

\[
\min_G \max_D V(D,G)
= \mathbb{E}[\log D(x \mid y)] + \mathbb{E}[\log(1 - D(G(z \mid y)))]
\]

Class embedding is concatenated (or projected) into \(G\) and \(D\).

### 1.4 Why GANs for *this* sparse-data problem

| Approach | Role |
|----------|------|
| **Classical augments** (rotate, blur, glare) | Cheap diversity from few seeds — always do this |
| **Procedural LCD renderer** | Bootstrap labelled ground-truth codes without real photos |
| **DCGAN / cGAN** | Learn residual distribution of textures, fonts, glare, sensor noise beyond hand-coded rules |
| **Diffusion (upgrade)** | Higher fidelity / stability for production OCR corpora |

Sparse-data pattern in this product's docs (`docs/24-…`): when ABox/evidence is thin, **synthetic completion + validation gates** beat pretending the graph is complete.

## Part 2 — Procedural seed images (LCD / 7-segment / matrix displays)

Real claim photos show codes on:

- dark LCD panels with grey segments,
- bright LED 7-segment modules,
- small monochrome OLED / matrix panels on appliance doors.

We simulate these **as labelled seeds**. GANs will later invent variations; OCR eval uses known labels.

In [ ]:
IMG_SIZE = 64  # DCGAN-friendly; increase to 128 if you have more train time / GPU


def _load_font(size: int) -> ImageFont.ImageFont:
    candidates = [
        "/System/Library/Fonts/Supplemental/Courier New Bold.ttf",
        "/System/Library/Fonts/Supplemental/Courier New.ttf",
        "/Library/Fonts/Courier New.ttf",
        "/System/Library/Fonts/Monaco.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
        "DejaVuSansMono-Bold.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size=size)
        except OSError:
            continue
    return ImageFont.load_default()


def render_lcd_panel(code: str, style: str = "lcd_dark", size: int = IMG_SIZE) -> Image.Image:
    """Render a single fault code as an appliance display crop."""
    code = code.upper()
    img = Image.new("RGB", (size, size))
    draw = ImageDraw.Draw(img)

    if style == "lcd_dark":
        bg, fg, bezel = (18, 28, 22), (140, 200, 120), (40, 40, 40)
    elif style == "lcd_blue":
        bg, fg, bezel = (10, 20, 50), (80, 160, 255), (30, 30, 45)
    elif style == "led_red":
        bg, fg, bezel = (8, 8, 8), (255, 40, 40), (20, 20, 20)
    elif style == "led_amber":
        bg, fg, bezel = (12, 10, 5), (255, 170, 40), (25, 22, 15)
    elif style == "oled_white":
        bg, fg, bezel = (5, 5, 5), (230, 230, 230), (15, 15, 15)
    else:
        bg, fg, bezel = (18, 28, 22), (140, 200, 120), (40, 40, 40)

    draw.rectangle([0, 0, size - 1, size - 1], fill=bezel)
    margin = max(4, size // 12)
    draw.rounded_rectangle(
        [margin, margin, size - 1 - margin, size - 1 - margin],
        radius=size // 16,
        fill=bg,
    )

    for y in range(margin + 2, size - margin - 2, 2):
        alpha = 8 if style.startswith("lcd") else 4
        draw.line(
            [(margin + 2, y), (size - margin - 3, y)],
            fill=tuple(min(255, c + alpha) for c in bg),
        )

    font_size = max(14, size // (2 if len(code) <= 2 else 3))
    font = _load_font(font_size)

    bbox = draw.textbbox((0, 0), code, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    x = (size - tw) // 2 - bbox[0]
    y = (size - th) // 2 - bbox[1] - size // 32
    if style.startswith("led"):
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (1, 1)]:
            draw.text((x + dx, y + dy), code, font=font, fill=tuple(c // 3 for c in fg))
    draw.text((x, y), code, font=font, fill=fg)

    if style in {"lcd_dark", "lcd_blue"} and size >= 64:
        small = _load_font(max(8, size // 10))
        label = "ERROR"
        bb = draw.textbbox((0, 0), label, font=small)
        lw = bb[2] - bb[0]
        draw.text(
            ((size - lw) // 2, margin + 2),
            label,
            font=small,
            fill=tuple(c // 2 for c in fg),
        )

    return img


STYLES = ["lcd_dark", "lcd_blue", "led_red", "led_amber", "oled_white"]

fig, axes = plt.subplots(len(STYLES), 6, figsize=(10, 8))
sample_codes = [FAULT_CATALOG[i]["code"] for i in range(6)]
for r, style in enumerate(STYLES):
    for c, code in enumerate(sample_codes):
        axes[r, c].imshow(render_lcd_panel(code, style=style, size=IMG_SIZE))
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(code, fontsize=9)
    axes[r, 0].set_ylabel(style, fontsize=8)
plt.suptitle("Procedural seed displays (domain fault codes)")
plt.tight_layout()
plt.savefig(ART / "preview_seed_styles.png", dpi=120)
plt.show()
print("Saved", ART / "preview_seed_styles.png")

In [ ]:
def build_seed_corpus(
    n_per_code: int = 40,
    size: int = IMG_SIZE,
    out_dir: Path = SEED_DIR,
) -> list[dict]:
    """Create labelled seed images on disk. Returns metadata rows."""
    meta: list[dict] = []
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in out_dir.glob("*.png"):
        p.unlink()

    for row in FAULT_CATALOG:
        code = row["code"]
        for i in range(n_per_code):
            style = STYLES[i % len(STYLES)]
            img = render_lcd_panel(code, style=style, size=size)
            factor = random.uniform(0.85, 1.15)
            img = ImageEnhance.Brightness(img).enhance(factor)
            fname = f"{code}__{style}__{i:03d}.png"
            path = out_dir / fname
            img.save(path)
            meta.append(
                {
                    "path": str(path),
                    "code": code,
                    "class_idx": CODE_TO_IDX[code],
                    "style": style,
                    "description": row["description"],
                    "family": row["family"],
                    "split": "seed",
                }
            )
    with open(ART / "seed_manifest.json", "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Wrote {len(meta)} seed images -> {out_dir}")
    return meta


seed_meta = build_seed_corpus(n_per_code=48)
print("Example:", seed_meta[0])

## Part 3 — Phone-camera augmentations (claim-photo realism)

Even without a GAN, classical augmentation multiplies sparse seeds (Shorten & Khoshgoftaar, 2019). We model:

- **Perspective / tilt** (phone not square to the panel)
- **Motion blur / defocus**
- **Glare / specular highlight**
- **ISO noise, JPEG-ish compression**
- **Partial crop / off-center framing**
- **Warm/cool white-balance shifts**

In [ ]:
def _random_glare(img: Image.Image) -> Image.Image:
    arr = np.array(img).astype(np.float32)
    h, w, _ = arr.shape
    cy, cx = random.uniform(0.2, 0.8) * h, random.uniform(0.2, 0.8) * w
    yy, xx = np.mgrid[0:h, 0:w]
    sigma = random.uniform(w * 0.08, w * 0.22)
    blob = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma**2))
    strength = random.uniform(40, 120)
    arr = np.clip(arr + (blob * strength)[..., None], 0, 255)
    return Image.fromarray(arr.astype(np.uint8))


def phone_camera_augment(img: Image.Image) -> Image.Image:
    """Map clean LCD seed to claim-photo-like crop."""
    out = img.convert("RGB")

    if random.random() < 0.9:
        out = out.rotate(random.uniform(-18, 18), resample=Image.BICUBIC, fillcolor=(30, 30, 30))

    if random.random() < 0.7:
        w, h = out.size
        dx = random.uniform(-0.08, 0.08) * w
        dy = random.uniform(-0.08, 0.08) * h
        out = out.transform(
            out.size,
            Image.AFFINE,
            (1, random.uniform(-0.12, 0.12), dx, random.uniform(-0.12, 0.12), 1, dy),
            resample=Image.BICUBIC,
            fillcolor=(25, 25, 25),
        )

    out = ImageEnhance.Brightness(out).enhance(random.uniform(0.55, 1.35))
    out = ImageEnhance.Contrast(out).enhance(random.uniform(0.7, 1.5))
    out = ImageEnhance.Color(out).enhance(random.uniform(0.6, 1.3))

    if random.random() < 0.55:
        out = out.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.4, 1.8)))

    if random.random() < 0.45:
        out = _random_glare(out)

    if random.random() < 0.7:
        arr = np.array(out).astype(np.float32)
        noise = np.random.normal(0, random.uniform(4, 18), arr.shape)
        out = Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

    if random.random() < 0.5:
        w, h = out.size
        scale = random.uniform(0.75, 0.95)
        nw, nh = int(w * scale), int(h * scale)
        left = random.randint(0, w - nw)
        top = random.randint(0, h - nh)
        out = out.crop((left, top, left + nw, top + nh)).resize((w, h), Image.BICUBIC)

    if random.random() < 0.25:
        out = ImageOps.posterize(out, bits=random.randint(5, 7))

    return out


demo = render_lcd_panel("5E", style="lcd_dark", size=IMG_SIZE)
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
axes[0, 0].imshow(demo)
axes[0, 0].set_title("seed")
axes[0, 0].axis("off")
for i in range(1, 12):
    r, c = divmod(i, 6)
    axes[r, c].imshow(phone_camera_augment(demo))
    axes[r, c].axis("off")
    axes[r, c].set_title(f"aug {i}")
plt.suptitle("Classical claim-photo augmentations of seed '5E'")
plt.tight_layout()
plt.savefig(ART / "preview_augments.png", dpi=120)
plt.show()

In [ ]:
def materialize_augmented(
    seed_meta: list[dict],
    n_aug_per_seed: int = 3,
    out_dir: Path = AUG_DIR,
) -> list[dict]:
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in out_dir.glob("*.png"):
        p.unlink()
    rows: list[dict] = []
    for s in seed_meta:
        base = Image.open(s["path"]).convert("RGB")
        dest = out_dir / Path(s["path"]).name
        base.save(dest)
        rows.append({**s, "path": str(dest), "split": "train", "source": "seed"})
        for j in range(n_aug_per_seed):
            aug = phone_camera_augment(base)
            fname = f"{s['code']}__aug__{Path(s['path']).stem}__{j}.png"
            path = out_dir / fname
            aug.save(path)
            rows.append(
                {
                    **{k: s[k] for k in ("code", "class_idx", "description", "family")},
                    "path": str(path),
                    "style": s.get("style", "aug"),
                    "split": "train",
                    "source": "augment",
                }
            )
    with open(ART / "train_manifest.json", "w") as f:
        json.dump(rows, f, indent=2)
    print(f"Train corpus: {len(rows)} images ({len(seed_meta)} seeds x ~{1 + n_aug_per_seed})")
    return rows


train_meta = materialize_augmented(seed_meta, n_aug_per_seed=2)


class FaultCodeDataset(Dataset):
    """Loads RGB images -> tensor in [-1, 1] for Tanh generators."""

    def __init__(self, meta: list[dict], image_size: int = IMG_SIZE, online_aug: bool = False):
        self.meta = meta
        self.online_aug = online_aug
        self.tf = transforms.Compose(
            [
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )

    def __len__(self) -> int:
        return len(self.meta)

    def __getitem__(self, idx: int):
        row = self.meta[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.online_aug and random.random() < 0.5:
            img = phone_camera_augment(img)
        x = self.tf(img)
        y = int(row["class_idx"])
        return x, y, row["code"]


dataset = FaultCodeDataset(train_meta, image_size=IMG_SIZE, online_aug=True)
loader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0, drop_last=True)
print(f"Dataset size={len(dataset)} | batches/epoch={len(loader)}")
xb, yb, codes = next(iter(loader))
print("batch", xb.shape, yb[:8].tolist(), codes[:8])

## Part 4 — DCGAN implementation

Architecture follows Radford et al. (2016) / [PyTorch DCGAN tutorial](https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html), sized for \(64 \times 64\) RGB.

In [ ]:
NZ = 100  # latent dim
NGF = 64  # generator feature maps
NDF = 64  # discriminator feature maps
NC = 3  # RGB


def weights_init(m: nn.Module) -> None:
    """DCGAN paper init: N(0, 0.02) for Conv; BN mean 1, std 0.02."""
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    """z (NZ,) -> RGB (NC, 64, 64)"""

    def __init__(self, nz: int = NZ, ngf: int = NGF, nc: int = NC):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class Discriminator(nn.Module):
    """RGB (NC, 64, 64) -> logit"""

    def __init__(self, ndf: int = NDF, nc: int = NC):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)


G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
G.apply(weights_init)
D.apply(weights_init)
print(G)
print(f"G params: {sum(p.numel() for p in G.parameters()):,}")
print(f"D params: {sum(p.numel() for p in D.parameters()):,}")

In [ ]:
# Increase EPOCHS (e.g. 50-100) for sharper digits after first smoke run.
EPOCHS = 12
LR = 2e-4
BETA1 = 0.5  # DCGAN paper
REAL_LABEL = 0.9  # one-sided label smoothing (Salimans et al. 2016 style)
FAKE_LABEL = 0.0

criterion = nn.BCEWithLogitsLoss()
opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, NZ, 1, 1, device=DEVICE)
history = {"loss_D": [], "loss_G": [], "D_x": [], "D_G_z": []}


def train_dcgan(epochs: int = EPOCHS):
    G.train()
    D.train()
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        for i, (real, _y, _codes) in enumerate(loader):
            bsz = real.size(0)
            real = real.to(DEVICE)

            D.zero_grad(set_to_none=True)
            label_real = torch.full((bsz,), REAL_LABEL, device=DEVICE)
            out_real = D(real)
            loss_D_real = criterion(out_real, label_real)

            noise = torch.randn(bsz, NZ, 1, 1, device=DEVICE)
            fake = G(noise)
            label_fake = torch.full((bsz,), FAKE_LABEL, device=DEVICE)
            out_fake = D(fake.detach())
            loss_D_fake = criterion(out_fake, label_fake)
            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            opt_D.step()

            G.zero_grad(set_to_none=True)
            out_fake_for_G = D(fake)
            loss_G = criterion(out_fake_for_G, label_real)
            loss_G.backward()
            opt_G.step()

            if i % 20 == 0:
                history["loss_D"].append(loss_D.item())
                history["loss_G"].append(loss_G.item())
                history["D_x"].append(torch.sigmoid(out_real).mean().item())
                history["D_G_z"].append(torch.sigmoid(out_fake_for_G).mean().item())

        print(
            f"[{epoch:03d}/{epochs}] "
            f"loss_D={loss_D.item():.3f} loss_G={loss_G.item():.3f} "
            f"D(x)={torch.sigmoid(out_real).mean().item():.3f} "
            f"D(G(z))={torch.sigmoid(out_fake_for_G).mean().item():.3f}"
        )

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            G.eval()
            with torch.no_grad():
                fake_snap = G(fixed_noise).detach().cpu()
            G.train()
            grid = make_grid(fake_snap, nrow=8, normalize=True, value_range=(-1, 1))
            save_image(grid, GEN_DIR / f"dcgan_epoch_{epoch:03d}.png")

    torch.save({"G": G.state_dict(), "D": D.state_dict(), "nz": NZ}, CKPT_DIR / "dcgan.pt")
    print(f"DCGAN done in {time.time() - t0:.1f}s -> {CKPT_DIR / 'dcgan.pt'}")


train_dcgan(EPOCHS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(history["loss_D"], label="D")
ax[0].plot(history["loss_G"], label="G")
ax[0].set_title("DCGAN losses")
ax[0].legend()
ax[0].set_xlabel("log step")
ax[1].plot(history["D_x"], label="D(real)")
ax[1].plot(history["D_G_z"], label="D(fake)")
ax[1].set_title("Discriminator outputs (~0.5 = confused)")
ax[1].legend()
plt.tight_layout()
plt.savefig(ART / "dcgan_training_curves.png", dpi=120)
plt.show()

try:
    from IPython.display import display
except ImportError:
    display = lambda x: None  # noqa: E731

last = sorted(GEN_DIR.glob("dcgan_epoch_*.png"))[-1]
print("Latest DCGAN samples:", last)
display(Image.open(last))

## Part 5 — Conditional GAN (generate a *chosen* fault code)

Unconditional DCGAN invents plausible panels but you cannot ask for `E24` specifically.
cGAN embeds the class index into \(G\) and \(D\) (Mirza & Osindero, 2014).

In [ ]:
class CondGenerator(nn.Module):
    def __init__(
        self,
        n_classes: int = N_CLASSES,
        nz: int = NZ,
        ngf: int = NGF,
        nc: int = NC,
        emb_dim: int = 50,
    ):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, emb_dim)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + emb_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        emb = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([z, emb], dim=1)
        return self.net(x)


class CondDiscriminator(nn.Module):
    def __init__(
        self,
        n_classes: int = N_CLASSES,
        ndf: int = NDF,
        nc: int = NC,
        emb_dim: int = 50,
    ):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, emb_dim)
        self.emb_dim = emb_dim
        self.net = nn.Sequential(
            nn.Conv2d(nc + emb_dim, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        emb = self.label_emb(labels)
        emb_map = emb.unsqueeze(2).unsqueeze(3).expand(b, self.emb_dim, h, w)
        x = torch.cat([x, emb_map], dim=1)
        return self.net(x).view(-1)


Gc = CondGenerator().to(DEVICE)
Dc = CondDiscriminator().to(DEVICE)
Gc.apply(weights_init)
Dc.apply(weights_init)
print(f"cGAN G params: {sum(p.numel() for p in Gc.parameters()):,}")
print(f"cGAN D params: {sum(p.numel() for p in Dc.parameters()):,}")

In [ ]:
CGAN_EPOCHS = 12  # raise to 40-80 for cleaner class-conditional digits
opt_Gc = torch.optim.Adam(Gc.parameters(), lr=LR, betas=(BETA1, 0.999))
opt_Dc = torch.optim.Adam(Dc.parameters(), lr=LR, betas=(BETA1, 0.999))
c_history = {"loss_D": [], "loss_G": []}

fixed_z = torch.randn(N_CLASSES * 4, NZ, 1, 1, device=DEVICE)
fixed_y = torch.arange(N_CLASSES, device=DEVICE).repeat_interleave(4)


def train_cgan(epochs: int = CGAN_EPOCHS):
    Gc.train()
    Dc.train()
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        for i, (real, labels, _codes) in enumerate(loader):
            bsz = real.size(0)
            real = real.to(DEVICE)
            labels = labels.to(DEVICE)

            Dc.zero_grad(set_to_none=True)
            loss_D_real = criterion(
                Dc(real, labels), torch.full((bsz,), REAL_LABEL, device=DEVICE)
            )
            z = torch.randn(bsz, NZ, 1, 1, device=DEVICE)
            fake_labels = (
                labels
                if random.random() < 0.7
                else torch.randint(0, N_CLASSES, (bsz,), device=DEVICE)
            )
            fake = Gc(z, fake_labels)
            loss_D_fake = criterion(
                Dc(fake.detach(), fake_labels),
                torch.full((bsz,), FAKE_LABEL, device=DEVICE),
            )
            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            opt_Dc.step()

            Gc.zero_grad(set_to_none=True)
            z = torch.randn(bsz, NZ, 1, 1, device=DEVICE)
            gen_labels = torch.randint(0, N_CLASSES, (bsz,), device=DEVICE)
            fake = Gc(z, gen_labels)
            loss_G = criterion(
                Dc(fake, gen_labels), torch.full((bsz,), REAL_LABEL, device=DEVICE)
            )
            loss_G.backward()
            opt_Gc.step()

            if i % 20 == 0:
                c_history["loss_D"].append(loss_D.item())
                c_history["loss_G"].append(loss_G.item())

        print(
            f"[cGAN {epoch:03d}/{epochs}] loss_D={loss_D.item():.3f} loss_G={loss_G.item():.3f}"
        )

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            Gc.eval()
            with torch.no_grad():
                snap = Gc(fixed_z, fixed_y).cpu()
            Gc.train()
            grid = make_grid(snap, nrow=4, normalize=True, value_range=(-1, 1))
            save_image(grid, GEN_DIR / f"cgan_epoch_{epoch:03d}.png")

    torch.save(
        {
            "G": Gc.state_dict(),
            "D": Dc.state_dict(),
            "nz": NZ,
            "n_classes": N_CLASSES,
            "idx_to_code": IDX_TO_CODE,
        },
        CKPT_DIR / "cgan.pt",
    )
    print(f"cGAN done in {time.time() - t0:.1f}s -> {CKPT_DIR / 'cgan.pt'}")


train_cgan(CGAN_EPOCHS)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(c_history["loss_D"], label="D")
ax.plot(c_history["loss_G"], label="G")
ax.set_title("cGAN losses")
ax.legend()
plt.tight_layout()
plt.savefig(ART / "cgan_training_curves.png", dpi=120)
plt.show()

last_c = sorted(GEN_DIR.glob("cgan_epoch_*.png"))[-1]
print("cGAN class grid (each row ~ one class x 4 samples):", last_c)
display(Image.open(last_c))
print("Row order:", [IDX_TO_CODE[i] for i in range(N_CLASSES)])

## Part 6 — Export synthetic claim-photo corpus

Generate a balanced set of images **per fault code** via cGAN, then apply phone-camera post-processing so OCR sees realistic noise.

In [ ]:
@torch.no_grad()
def generate_synthetic_corpus(
    n_per_code: int = 20,
    out_dir: Path | None = None,
    post_augment: bool = True,
) -> list[dict]:
    out_dir = out_dir or (GEN_DIR / "corpus")
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in out_dir.glob("*.png"):
        p.unlink()

    Gc.eval()
    rows: list[dict] = []
    for class_idx, code in IDX_TO_CODE.items():
        z = torch.randn(n_per_code, NZ, 1, 1, device=DEVICE)
        y = torch.full((n_per_code,), class_idx, device=DEVICE, dtype=torch.long)
        fakes = Gc(z, y).cpu()
        fakes = (fakes * 0.5 + 0.5).clamp(0, 1)
        for j in range(n_per_code):
            t = fakes[j]
            img = transforms.ToPILImage()(t)
            if post_augment:
                img = phone_camera_augment(img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))
            fname = f"synth__{code}__{j:03d}.png"
            path = out_dir / fname
            img.save(path)
            rows.append(
                {
                    "path": str(path),
                    "code": code,
                    "class_idx": class_idx,
                    "description": FAULT_CATALOG[class_idx]["description"],
                    "family": FAULT_CATALOG[class_idx]["family"],
                    "split": "synthetic",
                    "source": "cgan+phone_aug" if post_augment else "cgan",
                }
            )
    man = ART / "synthetic_corpus_manifest.json"
    with open(man, "w") as f:
        json.dump(rows, f, indent=2)
    print(f"Synthetic corpus: {len(rows)} images -> {out_dir}")
    print(f"Manifest: {man}")
    return rows


synth_meta = generate_synthetic_corpus(n_per_code=12, post_augment=True)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, row in zip(axes.ravel(), synth_meta[:16]):
    ax.imshow(Image.open(row["path"]))
    ax.set_title(row["code"], fontsize=8)
    ax.axis("off")
plt.suptitle("Synthetic claim-style fault-code images (cGAN + phone aug)")
plt.tight_layout()
plt.savefig(ART / "preview_synthetic_corpus.png", dpi=120)
plt.show()

## Part 7 — Extraction / OCR evaluation harness

Production systems typically use:

- commercial OCR (Azure Read, AWS Textract),
- `easyocr` / PaddleOCR / Tesseract,
- or a small fine-tuned CRNN / TrOCR on domain crops.

Here we implement a **domain-aware extractor** that:

1. prefers matches against the known catalog (closed-set codes),
2. falls back to alphanumeric regex,
3. scores accuracy on the labelled synthetic corpus (proxy metric until real photos exist).

> When real claim photos arrive: freeze a hold-out, re-run the same harness, and gate model promotion on real-image accuracy — never only synthetic (LLMOps / study module 20 guidance in this repo).

In [ ]:
# Closed-set catalog codes, longest first to avoid partials (e.g. F9E1 before E1)
_CATALOG_SORTED = sorted((row["code"].upper() for row in FAULT_CATALOG), key=len, reverse=True)
_CODE_RE = re.compile(r"\b([A-Z]{0,2}\d{1,2}[A-Z]{0,2}\d{0,2})\b")


def extract_fault_code_from_text(text: str) -> str | None:
    """Map free text / OCR string -> catalog code if present."""
    if not text:
        return None
    up = text.upper()
    for code in _CATALOG_SORTED:
        if re.search(rf"(?<![A-Z0-9]){re.escape(code)}(?![A-Z0-9])", up):
            return code
    m = _CODE_RE.search(up.replace("ERROR", " "))
    if m:
        cand = m.group(1)
        if cand in CODE_TO_IDX:
            return cand
    return None


def simple_pixel_template_ocr(
    img: Image.Image, templates: dict[str, Image.Image] | None = None
) -> str | None:
    """Ultra-light closed-set recognizer via NCC vs seed templates."""
    if templates is None:
        templates = {
            row["code"]: render_lcd_panel(row["code"], style="lcd_dark", size=IMG_SIZE)
            for row in FAULT_CATALOG
        }
    g = np.asarray(img.convert("L").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)
    g = (g - g.mean()) / (g.std() + 1e-6)
    best_code, best_score = None, -1e9
    for code, tmpl in templates.items():
        t = np.asarray(tmpl.convert("L").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)
        t = (t - t.mean()) / (t.std() + 1e-6)
        score = float((g * t).mean())
        if score > best_score:
            best_score, best_code = score, code
    return best_code


def tesseract_ocr(img: Image.Image) -> str | None:
    try:
        import pytesseract
        from pytesseract import TesseractNotFoundError
    except ImportError:
        return None
    try:
        big = img.resize((img.width * 4, img.height * 4), Image.BICUBIC)
        big = ImageOps.autocontrast(big.convert("L"))
        txt = pytesseract.image_to_string(
            big,
            config="--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
        )
        return extract_fault_code_from_text(txt)
    except (TesseractNotFoundError, OSError):
        # Package installed but binary missing, or OCR runtime failure
        return None


def extract_code_from_image(img: Image.Image) -> tuple[str | None, str]:
    """Returns (code, method)."""
    t = tesseract_ocr(img)
    if t:
        return t, "tesseract"
    c = simple_pixel_template_ocr(img)
    return c, "template_ncc"


def evaluate_extraction(meta: list[dict], max_n: int | None = 200) -> dict:
    rows = meta if max_n is None else meta[:max_n]
    correct = 0
    details = []
    for row in rows:
        img = Image.open(row["path"]).convert("RGB")
        pred, method = extract_code_from_image(img)
        ok = pred == row["code"]
        correct += int(ok)
        details.append(
            {
                "truth": row["code"],
                "pred": pred,
                "ok": ok,
                "method": method,
                "path": row["path"],
            }
        )
    acc = correct / max(len(rows), 1)
    report = {
        "n": len(rows),
        "correct": correct,
        "accuracy": acc,
        "method_mix": list({d["method"] for d in details}),
    }
    print(f"Extraction accuracy: {acc:.1%} ({correct}/{len(rows)}) methods={report['method_mix']}")
    misses = [d for d in details if not d["ok"]][:8]
    if misses:
        print("Sample misses:")
        for m in misses:
            print(f"  truth={m['truth']} pred={m['pred']} via {m['method']}")
    with open(ART / "ocr_eval_report.json", "w") as f:
        json.dump({"summary": report, "details": details}, f, indent=2)
    return report


clean_holdout = [m for m in seed_meta if m["style"] == "lcd_dark"][: N_CLASSES * 2]
print("--- Clean seeds ---")
evaluate_extraction(clean_holdout)
print("--- Synthetic claim-style ---")
evaluate_extraction(synth_meta)

## Part 8 — Bridge extracted codes → Neo4j Cypher (this codebase)

In production diagnose flow (`graph/graph_rag.py`):

1. `match_error_codes(product_id, user_message)` runs:

```cypher
MATCH (p:Product {product_id: $product_id})-[:HAS_ERROR_CODE]->(ec:ErrorCode)
RETURN ec.error_code_id AS error_code_id, ec.code AS code, ec.description AS description
```

…then keeps rows whose `code` appears in the (OCR-filled) message.

2. `rank_failure_modes_with_error_codes` **boosts** failure modes:

```cypher
MATCH (ec:ErrorCode)-[r:INDICATES]->(fm:FailureMode)
WHERE ec.error_code_id IN $error_code_ids
RETURN fm.failure_mode_id AS failure_mode_id, sum(r.confidence) AS error_boost
```

### End-to-end simulation (no live Neo4j required)

We turn an image → code → **parameterized Cypher** + a local mini-graph that mirrors the ontology edges.

In [ ]:
# Mini ABox mirror of catalog relationships (illustrative; production = Neo4j)
LOCAL_GRAPH = {
    "UE": {"failure_modes": [("unbalanced_load", 0.75)], "steps": ["Redistribute load", "Run calibration spin"]},
    "4E": {"failure_modes": [("inlet_valve_fault", 0.68)], "steps": ["Check water supply taps", "Inspect inlet screens"]},
    "5E": {
        "failure_modes": [("drain_pump_block", 0.92)],
        "steps": ["Clean drain filter", "Inspect drain hose"],
        "parts": ["drain_pump", "drain_filter"],
    },
    "OE": {"failure_modes": [("drain_timeout", 0.90)], "steps": ["Verify standpipe height", "Check pump"]},
    "IE": {"failure_modes": [("fill_insufficient", 0.93)], "steps": ["Confirm inlet pressure", "Test pressure sensor"]},
    "HE": {"failure_modes": [("heater_fault", 0.92)], "steps": ["Measure heater resistance", "Check NTC sensor"]},
    "AE": {"failure_modes": [("leak_detected", 0.55)], "steps": ["Inspect base tray", "Check hoses"]},
    "E24": {"failure_modes": [("drain_issue", 0.95)], "steps": ["Clear filter", "Test drain pump"]},
    "E01": {"failure_modes": [("pump_failure", 0.96)], "steps": ["Ohm-test pump", "Replace if open circuit"]},
    "F9E1": {"failure_modes": [("drain_pump_fault", 0.96)], "steps": ["Replace drain pump", "Verify harness"]},
    "F9E0": {"failure_modes": [("drain_system_fault", 0.90)], "steps": ["Inspect drain path"]},
    "22E": {"failure_modes": [("evap_fan_error", 0.96)], "steps": ["Test evaporator fan"]},
    "33E": {"failure_modes": [("ice_fan_error", 0.95)], "steps": ["Test ice-maker fan"]},
    "39E": {"failure_modes": [("ice_function_error", 0.90)], "steps": ["Run ice-maker diagnostics"]},
    "LC": {"failure_modes": [("control_lock", 0.99)], "steps": ["Hold temp buttons 3s to unlock"]},
    "OC": {"failure_modes": [("overload", 0.85)], "steps": ["Reduce load", "Inspect motor current"]},
    "DE": {"failure_modes": [("door_error", 0.90)], "steps": ["Check door switch / latch"]},
}


def cypher_for_extracted_code(code: str, product_id: str = "wm-001") -> dict:
    """Return the Cypher statements our GraphRAG layer effectively needs."""
    q_match = """
MATCH (p:Product {product_id: $product_id})-[:HAS_ERROR_CODE]->(ec:ErrorCode)
WHERE toUpper(ec.code) = toUpper($code)
RETURN ec.error_code_id AS error_code_id,
       ec.code AS code,
       ec.description AS description
""".strip()
    q_fm = """
MATCH (p:Product {product_id: $product_id})-[:HAS_ERROR_CODE]->(ec:ErrorCode)
WHERE toUpper(ec.code) = toUpper($code)
MATCH (ec)-[r:INDICATES]->(fm:FailureMode)
RETURN fm.failure_mode_id AS failure_mode_id,
       fm.name AS name,
       r.confidence AS confidence
ORDER BY r.confidence DESC
""".strip()
    q_res = """
MATCH (p:Product {product_id: $product_id})-[:HAS_ERROR_CODE]->(ec:ErrorCode)
WHERE toUpper(ec.code) = toUpper($code)
MATCH (ec)-[:INDICATES]->(fm:FailureMode)
OPTIONAL MATCH (ds:DiagnosticStep)-[c:CONFIRMS]->(fm)
OPTIONAL MATCH (fm)-[:REQUIRES_PART]->(part:Part)
OPTIONAL MATCH (hr:HistoricalResolution)-[:RESOLVED]->(fm)
RETURN ec, fm,
       collect(DISTINCT ds) AS confirm_steps,
       collect(DISTINCT part) AS parts,
       collect(DISTINCT hr) AS historical
""".strip()
    return {
        "match_code_on_product": q_match,
        "failure_modes_for_code": q_fm,
        "resolution_path": q_res,
        "params": {"product_id": product_id, "code": code},
    }


def diagnose_from_image(path, product_id: str = "wm-001") -> dict:
    img = Image.open(path).convert("RGB")
    code, method = extract_code_from_image(img)
    result = {
        "image": str(path),
        "extracted_code": code,
        "ocr_method": method,
        "product_id": product_id,
        "graph_hit": None,
        "cypher": None,
        "user_message_for_api": None,
    }
    if not code:
        result["error"] = "No fault code extracted — escalate or request retake photo"
        return result

    # Message format that match_error_codes() in graph_rag.py can consume
    result["user_message_for_api"] = (
        f"Customer uploaded display photo. Error code {code} shown on machine."
    )
    result["cypher"] = cypher_for_extracted_code(code, product_id=product_id)
    local = LOCAL_GRAPH.get(code, {})
    result["graph_hit"] = {
        "code": code,
        "description": next((r["description"] for r in FAULT_CATALOG if r["code"] == code), ""),
        "failure_modes": local.get("failure_modes", []),
        "diagnostic_steps": local.get("steps", []),
        "parts": local.get("parts", []),
        "note": "Local mirror for offline demo; production uses Neo4j INDICATES/CONFIRMS edges",
    }
    return result


demo_paths = [
    next(r["path"] for r in seed_meta if r["code"] == c and r["style"] == "lcd_dark")
    for c in ("5E", "UE", "E24", "F9E1")
]

for p in demo_paths:
    diag = diagnose_from_image(p, product_id="wm-001")
    print("=" * 72)
    print("IMAGE:", Path(diag["image"]).name)
    print("EXTRACTED:", diag["extracted_code"], f"({diag['ocr_method']})")
    print("API MESSAGE:", diag.get("user_message_for_api"))
    if diag.get("graph_hit"):
        print("FM:", diag["graph_hit"]["failure_modes"])
        print("STEPS:", diag["graph_hit"]["diagnostic_steps"])
    if diag.get("cypher"):
        print("--- Cypher: failure_modes_for_code ---")
        print(diag["cypher"]["failure_modes_for_code"])
        print("params:", diag["cypher"]["params"])

In [ ]:
# Optional: live Neo4j query if the dual-graph demo stack is up
# (docker compose from repo — production bolt usually localhost:7687)


def try_live_neo4j(code: str, product_id: str = "wm-001"):
    try:
        from neo4j import GraphDatabase
    except ImportError:
        print("neo4j driver not installed")
        return None
    uri = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
    user = os.environ.get("NEO4J_USER", "neo4j")
    password = os.environ.get("NEO4J_PASSWORD", "password")
    query = (
        "MATCH (p:Product {product_id: $product_id})-[:HAS_ERROR_CODE]->(ec:ErrorCode) "
        "WHERE toUpper(ec.code) = toUpper($code) "
        "OPTIONAL MATCH (ec)-[r:INDICATES]->(fm:FailureMode) "
        "RETURN ec.code AS code, ec.description AS description, "
        "collect({fm: fm.failure_mode_id, conf: r.confidence}) AS failure_modes"
    )
    try:
        driver = GraphDatabase.driver(uri, auth=(user, password))
        with driver.session() as session:
            rec = session.run(query, product_id=product_id, code=code).single()
        driver.close()
        if rec is None:
            print(
                f"No live hit for product={product_id} code={code} "
                "(graph empty or wrong product_id)"
            )
            return None
        data = dict(rec)
        print("Live Neo4j hit:", data)
        return [data]
    except Exception as exc:  # noqa: BLE001 — demo optional path
        print(f"Live Neo4j unavailable ({exc}). Offline Cypher templates above still apply.")
        return None


# Uncomment when Neo4j is running and product ABox is loaded:
# try_live_neo4j("5E", product_id="wm-001")
print("Live Neo4j helper defined as try_live_neo4j(code, product_id).")

## Part 8b — How this plugs into the diagnose API

Conceptual integration (does not change production code in this notebook):

```text
POST /diagnose  (or future POST /claims/{id}/attachments)
  multipart: image + product_id / asset_id
       |
       v
  crop panel ROI -> extract_code_from_image()
       |
       v
  user_message += " Error code {code} shown on machine."
       |
       v
  services/diagnosis_service.py -> graph_rag.match_error_codes()
       |
       v
  rank_failure_modes_with_error_codes()  # INDICATES confidence boost
       |
       v
  CONFIRMS diagnostic steps + parts + historical resolutions
```

Synthetic images from Part 6 populate:

| Artifact | Use |
|----------|-----|
| `synthetic_corpus_manifest.json` | Train / fine-tune OCR or vision model |
| `ocr_eval_report.json` | Efficacy baseline (synthetic proxy) |
| `checkpoints/cgan.pt` | Regenerate more images on demand for rare codes |

### Production hardening checklist

1. **Real-photo hold-out gate** before promoting OCR models.
2. **PII / scene safety** — claim photos may include faces, rooms; redact if stored.
3. **Closed-set preference** — only accept codes in product's `HAS_ERROR_CODE` list (prevents OCR hallucinations from inventing edges).
4. **Confidence + human escalate** when OCR confidence low (matches existing `should_escalate` patterns).
5. **Lineage** — tag synthetic samples so they never silently mix into "historical claim photo" truth tables without label.

## Part 9 — References & further reading

### Foundational papers
1. **I. Goodfellow et al.** — *Generative Adversarial Nets*. NeurIPS 2014.
   https://papers.nips.cc/paper/5423-generative-adversarial-nets
2. **A. Radford, L. Metz, S. Chintala** — *Unsupervised Representation Learning with Deep Convolutional GANs*. ICLR 2016 workshop / arXiv:1511.06434.
3. **M. Mirza, S. Osindero** — *Conditional Generative Adversarial Nets*. arXiv:1411.1784, 2014.
4. **T. Salimans et al.** — *Improved Techniques for Training GANs*. NeurIPS 2016 (label smoothing, feature matching).
5. **I. Gulrajani et al.** — *Improved Training of Wasserstein GANs* (WGAN-GP). NeurIPS 2017 — if mode collapse appears.
6. **C. Shorten, T. Khoshgoftaar** — *A survey on Image Data Augmentation for Deep Learning*. Journal of Big Data, 2019.

### Tutorials & notebooks (implementations)
- PyTorch official DCGAN faces tutorial: https://pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html
- PyTorch examples `dcgan`: https://github.com/pytorch/examples/tree/main/dcgan
- Conditional GAN pattern: class embedding concatenated into G/D (Part 5 of this notebook).
- Higher-fidelity **upgrade path**: DDPM / latent diffusion (Ho et al. 2020; Rombach et al. LDM 2022).

### This repository
- `graph/graph_rag.py` — `match_error_codes`, `rank_failure_modes_with_error_codes`
- `graph/oem_product_catalog.py` — authoritative demo fault codes & INDICATES links
- `docs/24-TBox-Sparse-Data-and-Enterprise-Scale-Patterns.md` — sparse evidence philosophy
- Study module "Synthetic data, MLOps, image generation concepts" (`study/seed_platform_modules.py`)

### Suggested next experiments

| Experiment | Why |
|------------|-----|
| Train longer cGAN (50–100 epochs) or raise `IMG_SIZE` to 128 | Sharper glyphs for OCR |
| WGAN-GP loss | Stability if \(D\) overpowers \(G\) |
| Replace template OCR with EasyOCR / TrOCR fine-tune on `synthetic_corpus` | Production-grade extraction |
| Domain-randomize backgrounds (kitchen, laundry room) via composition | Closer to true claim photos |
| Active learning: prioritize rare codes (`F9E1`) for synthetic oversampling | Class imbalance in claims |

---

### Artifacts written by this notebook

```text
notebooks/fault_code_gan_artifacts/
  seed_lcd/                  # procedural LCD seeds
  augmented/                 # seeds + classical augments (GAN train set)
  generated/                 # DCGAN/cGAN grids + corpus/
  checkpoints/dcgan.pt
  checkpoints/cgan.pt
  seed_manifest.json
  train_manifest.json
  synthetic_corpus_manifest.json
  ocr_eval_report.json
  preview_*.png
```

**Done when:** you can generate labelled claim-style images for every catalog code, run the extraction harness, and emit Cypher that matches this product's GraphRAG contracts.